# Modul 5: Persona & Workspace — Den Agent formen

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook definierst du Persona-Templates und baust einen Konsistenz-Checker.

🎯 **Lernziel:** Persona als System-Prompt verstehen und Konsistenz über 10 Prompts messen.

---

## Persona = Die "Verfassung" deines Agents

Eine Persona definiert:
- **Identität:** Wer bin ich? (Name, Rolle)
- **Stil:** Wie kommuniziere ich? (Sprache, Tonfall, Länge)
- **Grenzen:** Was mache ich NICHT? (Ethik, Datenschutz)
- **Regeln:** Wie verhalte ich mich in Sonderfällen?

Shanahan et al. (2023) zeigen: **Konsistente Rollen-Instruktionen verbessern die Kohärenz** über mehrere Gesprächs-Turns.

```
┌──────────────────────────────────┐
│          WORKSPACE FILES         │
│                                  │
│  SOUL.md     → Persönlichkeit    │
│  IDENTITY.md → Name & Rolle      │
│  USER.md     → Nutzerpräferenzen │
│  AGENTS.md   → Verhaltensregeln  │
└──────────────────────────────────┘
```

In [ ]:
# Persona-Templates als Python-Dictionaries

# Template 1: Professioneller Assistent
persona_professional = {
    'name': 'Atlas',
    'role': 'Professioneller Forschungsassistent',
    'language': 'deutsch',
    'style': {
        'tone': 'sachlich und präzise',
        'length': 'kurz (1-3 Sätze)',
        'formality': 'Sie-Form',
    },
    'rules': [
        'Immer Quellen angeben',
        'Keine Spekulationen ohne Kennzeichnung',
        'Bei Unsicherheit nachfragen',
    ],
    'boundaries': [
        'Keine medizinischen Diagnosen',
        'Keine rechtliche Beratung',
        'Keine persönlichen Daten speichern',
    ]
}

# Template 2: Lockerer Buddy
persona_casual = {
    'name': 'Buddy',
    'role': 'Lockerer Technik-Freund',
    'language': 'deutsch',
    'style': {
        'tone': 'freundlich und locker',
        'length': 'kurz, mit Emojis',
        'formality': 'Du-Form',
    },
    'rules': [
        'Emojis nutzen',
        'Technische Begriffe einfach erklären',
        'Humor ist erlaubt',
    ],
    'boundaries': [
        'Kein Sarkasmus bei ernsten Themen',
        'Keine Abkürzungen ohne Erklärung',
    ]
}

# Template 3: Strenger Sicherheitsexperte
persona_security = {
    'name': 'Guardian',
    'role': 'Security-Analyst',
    'language': 'deutsch',
    'style': {
        'tone': 'ernst und warnend',
        'length': 'detailliert bei Risiken',
        'formality': 'neutral',
    },
    'rules': [
        'Risiken immer zuerst nennen',
        'Best Practices als Checkliste',
        'Niemals Passwörter oder Keys ausgeben',
    ],
    'boundaries': [
        'Keine Hacking-Anleitungen',
        'Keine Umgehung von Sicherheitsmaßnahmen',
    ]
}


def render_system_prompt(persona):
    """Wandelt ein Persona-Dict in einen System-Prompt um."""
    lines = []
    lines.append(f'Du bist {persona["name"]} — {persona["role"]}.')
    lines.append(f'Sprache: {persona["language"]}.')
    lines.append(f'Ton: {persona["style"]["tone"]}. Antwortlänge: {persona["style"]["length"]}.')
    lines.append(f'Anrede: {persona["style"]["formality"]}.')
    
    if persona['rules']:
        lines.append('\nRegeln:')
        for r in persona['rules']:
            lines.append(f'- {r}')
    
    if persona['boundaries']:
        lines.append('\nGrenzen (NIEMALS):')
        for b in persona['boundaries']:
            lines.append(f'- {b}')
    
    return '\n'.join(lines)


# System-Prompts rendern
for persona in [persona_professional, persona_casual, persona_security]:
    print(f'=== {persona["name"]} ===')
    print(render_system_prompt(persona))
    print()

In [ ]:
# 10-Prompt Konsistenz-Checker
# Prüft ob simulierte Antworten zur Persona passen

def simulate_response(persona, prompt):
    """Simuliert eine Antwort basierend auf der Persona (ohne LLM)."""
    style = persona['style']
    name = persona['name']
    
    # Simulierte Antworten je nach Persona-Stil
    if style['formality'] == 'Sie-Form':
        greeting = 'Guten Tag'
        pronoun = 'Sie'
    elif style['formality'] == 'Du-Form':
        greeting = 'Hey'
        pronoun = 'du'
    else:
        greeting = 'Hallo'
        pronoun = 'man'
    
    if 'wetter' in prompt.lower():
        base = f'{greeting}! Das Wetter ist sonnig, 18°C.'
    elif 'hilfe' in prompt.lower():
        base = f'{greeting}! Klar, ich helfe {pronoun} gerne.'
    elif 'gefährlich' in prompt.lower() or 'risiko' in prompt.lower():
        base = f'{greeting}. ⚠️ Vorsicht — das birgt Risiken.'
    elif 'wer bist du' in prompt.lower():
        base = f'{greeting}! Ich bin {name}, {persona["role"]}.'
    else:
        base = f'{greeting}! Interessante Frage.'
    
    # Emoji für lockere Persona
    if 'Emojis nutzen' in persona.get('rules', []):
        base += ' 😊'
    
    return base


def check_consistency(persona, test_prompts):
    """Prüft Konsistenz über mehrere Prompts via String-Matching."""
    results = []
    
    checks = {
        'name_used': lambda r: persona['name'].lower() in r.lower(),
        'correct_formality': lambda r: (
            ('Sie' in r if persona['style']['formality'] == 'Sie-Form'
             else 'du' in r.lower() if persona['style']['formality'] == 'Du-Form'
             else True)
        ),
        'has_emoji': lambda r: any(c for c in r if ord(c) > 0x1F600),
        'not_too_long': lambda r: len(r) < 200,  # Kurze Antworten
    }
    
    print(f'\n🔍 Konsistenz-Check für "{persona["name"]}"')
    print(f'{"Prompt":<30} {"Antwort":<50} {"Score"}')
    print('-' * 90)
    
    total_score = 0
    for prompt in test_prompts:
        response = simulate_response(persona, prompt)
        
        # Relevante Checks für diese Persona
        score = 0
        max_score = 0
        
        # Formalität prüfen
        max_score += 1
        if checks['correct_formality'](response):
            score += 1
        
        # Länge prüfen
        max_score += 1
        if checks['not_too_long'](response):
            score += 1
        
        # Emoji-Check (nur wenn erwartet)
        if 'Emojis nutzen' in persona.get('rules', []):
            max_score += 1
            if checks['has_emoji'](response):
                score += 1
        
        pct = score / max_score if max_score > 0 else 0
        icon = '✅' if pct >= 0.5 else '❌'
        print(f'{prompt:<30} {response:<50} {icon} {score}/{max_score}')
        total_score += pct
    
    avg = total_score / len(test_prompts) * 100
    print(f'\n📊 Gesamtkonsistenz: {avg:.0f}% ({"BESTANDEN ✅" if avg >= 80 else "DURCHGEFALLEN ❌"})')
    return avg


# 10 Test-Prompts
test_prompts = [
    'Wer bist du?',
    'Wie wird das Wetter?',
    'Hilfe bitte!',
    'Was ist Python?',
    'Ist das gefährlich?',
    'Erkläre mir Machine Learning',
    'Guten Morgen!',
    'Was ist ein Agent?',
    'Hilfe bei meinem Projekt',
    'Gibt es Risiken?',
]

# Alle Personas testen
for persona in [persona_professional, persona_casual, persona_security]:
    check_consistency(persona, test_prompts)

In [ ]:
# 🎯 ÜBUNG: Erstelle deine eigene Persona
#
# Aufgabe:
# 1. Definiere ein persona-Dictionary für DEINEN idealen Agent
# 2. Gib ihm einen Namen, Rolle, Stil-Regeln und Grenzen
# 3. Rendere den System-Prompt mit render_system_prompt()
# 4. Teste die Konsistenz mit check_consistency()

my_persona = {
    'name': '',      # TODO: Name vergeben
    'role': '',      # TODO: Rolle definieren
    'language': 'deutsch',
    'style': {
        'tone': '',       # TODO: Ton festlegen
        'length': '',     # TODO: Antwortlänge
        'formality': '',  # TODO: 'Du-Form', 'Sie-Form' oder 'neutral'
    },
    'rules': [
        # TODO: 2-3 Regeln
    ],
    'boundaries': [
        # TODO: 1-2 Grenzen
    ]
}

# TODO: System-Prompt rendern
# print(render_system_prompt(my_persona))

# TODO: Konsistenz testen
# check_consistency(my_persona, test_prompts)

In [ ]:
# ✅ LÖSUNG (Beispiel-Persona)

my_persona = {
    'name': 'Planck',
    'role': 'Persönlicher Technik-Assistent',
    'language': 'deutsch',
    'style': {
        'tone': 'freundlich aber präzise',
        'length': 'kurz (1-3 Sätze)',
        'formality': 'Du-Form',
    },
    'rules': [
        'Immer auf Deutsch antworten',
        'Bei Unsicherheit kurz nachfragen',
        'Technische Fachbegriffe auf Englisch lassen',
    ],
    'boundaries': [
        'Keine Arbeitsdaten verarbeiten',
        'Keine API-Keys oder Passwörter ausgeben',
    ]
}

print('=== Meine Persona ===')
print(render_system_prompt(my_persona))
print()
check_consistency(my_persona, test_prompts)

## Persona in der Praxis

| Datei | Inhalt | Beispiel |
|-------|--------|----------|
| `SOUL.md` | Persönlichkeit, Stil, Grenzen | "Du bist Planck, freundlich und präzise" |
| `IDENTITY.md` | Name, Rolle, Emoji | "Planck, ⚛️" |
| `USER.md` | Nutzer-Präferenzen | "Jan, Zeitzone Berlin, Deutsch" |
| `AGENTS.md` | Verhaltensregeln, Routing | Security-Rules, Skill-Tabelle |

---

### ✅ Self-Check
- [ ] Ich kann eine Persona als Dictionary definieren
- [ ] Ich verstehe warum Konsistenz-Tests wichtig sind (≥8/10)
- [ ] Ich kann den Unterschied zwischen Stil und Grenzen erklären

**→ Weiter zu [Modul 6: Gedächtnis & Kontext](modul6_gedaechtnis.ipynb)**